# 04 - Model comparison
Reads the saved results of a finished experiment (`outputs/<run>/`). Nothing is retrained here.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
from src.config import load_config
cfg = load_config()

In [ ]:
import json
runs = sorted((ROOT / "outputs").glob("*/summary.json"))
print([r.parent.name for r in runs])
run = next((r for r in runs if r.parent.name == "final_v1"), runs[-1])
summary = json.loads(run.read_text())
print("run:", run.parent.name, "| status:", summary["status"])
res = pd.read_csv(run.parent / "results_per_split.csv")
res.groupby(["protocol", "model"])[["accuracy", "balanced_accuracy", "f1", "roc_auc"]].agg(["mean", "std"]).round(3)

In [ ]:
for protocol, g in res.groupby("protocol"):
    piv = g.pivot(index="test_subject", columns="model", values="balanced_accuracy")
    ax = piv.plot.bar(figsize=(9, 3.5), title=f"Balanced accuracy per held-out participant ({protocol})")
    ax.axhline(0.5, color="k", ls="--", lw=0.8); ax.set_ylabel("Balanced accuracy"); ax.set_ylim(0, 1)

In [ ]:
print(json.dumps({p: v.get("paired_comparisons") for p, v in summary["aggregate"].items()}, indent=2))

Paired comparisons use nine participants only; p-values are indicative at best. See `docs/EXPERIMENTS.md` and `docs/LIMITATIONS.md`.